In [85]:
import os
from datetime import datetime

import json
from tqdm.auto import tqdm
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI

In [3]:
ROOT_DIR = "/data/ephemeral/pro-nlp-finalproject-nlp-13/data/"
dir = os.path.join(ROOT_DIR, 'ko-freshqa_2025_dev.csv')
df = pd.read_csv(dir, index_col=False)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              550 non-null    int64  
 1   split           550 non-null    object 
 2   domain          550 non-null    object 
 3   question        550 non-null    object 
 4   effective_year  550 non-null    object 
 5   next_review     283 non-null    object 
 6   false_premise   550 non-null    bool   
 7   num_hops        550 non-null    object 
 8   fact_type       550 non-null    object 
 9   source          550 non-null    object 
 10  answer_0        550 non-null    object 
 11  answer_1        550 non-null    object 
 12  answer_2        134 non-null    object 
 13  answer_3        11 non-null     object 
 14  answer_4        2 non-null      object 
 15  answer_5        0 non-null      float64
 16  answer_6        0 non-null      float64
 17  answer_7        0 non-null      flo

In [8]:
load_dotenv()

True

In [17]:
client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)

In [33]:
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "임수정 배우의 나이는?"
        }
    ],
    reasoning_effort="high"
)

In [34]:
print(response.choices[0].message.content.strip())

임수정 배우의 나이는 **1981년 1월 6일**생으로, 2023년 기준 **42세**입니다. 2024년 현재(만 나이 적용 시) **43세**로 계산됩니다. 

한국식 나이(세는 나이)로는 출생 시점에 1살로 계산되므로, 2024년 현재 **44세**가 될 수 있으나, 일반적으로 국제적 기준(만 나이)을 따르는 경우가 많습니다. 최신 정보를 확인하려면 공식 프로필 또는 신뢰할 수 있는 언론 자료를 참고하는 것이 좋습니다.


In [35]:
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "임수정 배우의 나이는?"
        }
    ],
)

In [36]:
print(response.choices[0].message.content.strip())

임수정 배우의 나이는 **1981년 1월 21일**생으로, **2024년 기준 43세**입니다. 한국 나이로는 44세입니다.  

참고로, 임수정은 영화 **《마녀》**, **《타짜: 원 아이드 잭》**, 드라마 **《보이스》**, **《미스터 로봇》** 등에 출연하며 활발히 활동 중인 배우입니다. 정확한 생일을 고려할 경우, 생일이 지나지 않았다면 한국식 나이는 한 살 적을 수 있습니다.  

최신 정보는 공식 프로필 또는 언론 인터뷰를 참고하시는 것이 정확합니다.


In [41]:
def ask_solar_pro2(message):
    response = client.chat.completions.create(
        model="solar-pro2",
        messages=[
            {
                "role": "user",
                "content": message
            }
        ],
        #reasoning_effort="high",
    )
    return response.choices[0].message.content.strip()

In [42]:
ask_solar_pro2("세종대왕이 쓴 아이폰 모델명이 뭐야?")

'세종대왕은 15세기 조선의 왕으로서, 아이폰(21세기 애플사의 스마트폰)과는 시대적 배경이 완전히 다릅니다. 따라서 세종대왕이 쓴 아이폰 모델은 존재하지 않습니다.  \n\n이 질문은 역사적 사실과 기술을 유머러스하게 결합한 말장난으로 보입니다.  \n혹시 다른 궁금한 점이 있다면 알려주세요! 😊'

### 초기 답변 도출
- LLM이 답변을 도출하도록 설계
- 답변을 짧게,
- TODAY가 필요하게

In [ ]:
PROMPT = """
당신은 세계 최고의 지식 전문가이자 논리적인 분석가입니다.
사용자의 질문에 대해 검색 없이, TODAY 기준으로 가장 안전하고 정확한 답을 만듭니다.
반드시 순수 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)

### [기준 정보]
- TODAY: {TODAY}
- 모든 판단은 TODAY 기준으로 과거/현재/미래를 구분합니다.

### [출력 형식]
- 반드시 아래 두 개의 키를 가진 순수 JSON만 출력하세요.
  1. "thought_process": 질문의 전제 검증 및 단계별 사고 과정 (한국어)
  2. "answer": 최종 정답(초단답)

### [답변 작성 지침]
1. **thought_process (사고 과정)**:
   - **전제 검증:** 질문에 사실과 다른 전제(False Premise)가 있는지 확인하세요. (예: "이순신의 아이폰" -> "조선시대엔 아이폰 없음")
   - **시점 확인:** 질문 내용이 현재({TODAY})보다 미래인지 확인하세요. 미래라면 "알 수 없음"으로 판단하세요.
   - **목록 나열:** '몇 개', '누구' 등 개수나 목록을 묻는 질문은 **반드시 기억나는 대상을 먼저 리스트업**한 뒤 숫자를 세세요.

2. **final_answer (최종 정답)**:
   - `thought_process`에서 내린 결론을 사용자에게 전달할 **최종 문장**으로 작성하세요.
   - **[주의] 메타 발언 금지:** "전제가 틀렸습니다", "제 생각에는", "질문을 수정합니다" 같은 설명조의 말을 **절대** 쓰지 마세요.
   - **[작성 법]** 바로 수정된 사실이나 정답만 건조하게 서술하세요.
     - (Bad): "전제가 틀렸습니다. 한국에서 비트코인은 법정화폐가 아닙니다."
     - (Good): "한국에서 비트코인은 법정화폐로 인정되지 않았습니다."

### [출력 예시]
{
    "thought_process": "사용자가 비트코인 법정화폐 인정 이유를 묻고 있다. -> 현재 날짜 기준 한국은 비트코인을 법정화폐로 인정한 적이 없다. -> 따라서 이유는 존재하지 않으며, 인정되지 않았다는 사실을 전달해야 한다.",
    "answer": "한국에서 비트코인은 법정화폐로 인정되지 않았습니다."
}
"""

In [57]:
SYSTEM_PROMPT1 = """
당신은 세계 최고의 지식 전문가이자 논리적인 분석가입니다.
사용자의 질문에 대해 검색 없이, TODAY 기준으로 가장 안전하고 정확한 답을 만듭니다.
반드시 순수 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)

### [기준 정보]
- TODAY: {TODAY}

### [출력 형식]
- 반드시 아래 두 개의 키를 가진 순수 JSON만 출력하세요.
  1. "thought_process": 아래 [사고 과정]을 거친 논리적 분석 (한국어)
  2. "answer": 최종 정답 (간결하고 직접적인 문장)

### [사고 과정(필수 절차)]
thought_process에서 다음 단계를 엄격히 수행하세요:
1) **전제 검증(Premise Check):** - 질문에 포함된 인물, 단체, 사건, 시간이 실제 역사나 팩트와 일치하는가?
   - 잘못된 전제 예시: 존재하지 않는 직책(예: 한국 국왕), 시간적 모순(예: 조선시대 아이폰), 사실 왜곡(예: 2022 월드컵 우승국은 한국)
2) **시점 확인:** TODAY({TODAY}) 기준 미래의 일인가? 혹은 아직 결과가 나오지 않은 미정 사항인가?
3) **지식 경계 확인:** 내 학습 데이터 내에서 100% 확신 가능한가? (모르면 환각 금지)
4) **대응 설계:** 전제 오류 시 교정형 답변, 미래/불확실 정보 시 유보형 답변 선택.

### [환각 방지 강제 조항]
1. **지식의 한계 준수:** 답변을 위해 새로운 사실, 숫자, 날짜를 발명하지 마세요. 100% 확신이 없는 고유명사나 통계는 절대 지어내지 않습니다.
2. **모름의 미학:** 오답(환각)을 내는 것보다 "확인되지 않습니다"라고 답하는 것이 리더보드 점수 방어에 유리함을 명심하세요.
3. **최신성 경계:** TODAY({TODAY})와 너무 가까운 시점의 실시간 정보(주가, 당일 뉴스 등)는 "알 수 없음"으로 처리합니다.

### [답변 규칙]
- **전제 오류(False Premise) 대응:**
  - 질문의 틀린 부분을 직접적으로 바로잡는 팩트를 제시하세요.
  - (Bad): "질문이 틀렸습니다. 이순신은 아이폰을 안 썼습니다."
  - (Good): "이순신 장군이 생존했던 시기에는 아이폰이 존재하지 않았습니다."
- **최신/변동 정보 대응:** - TODAY 기준 확인이 필요한 최신 사실(스포츠 결과, 경제 수치 등)은 함부로 단정하지 말고 "확인되지 않았습니다" 혹은 "발표된 바 없습니다"라고 답하세요.
- **메타 발언 금지:** "제 생각에는", "분석 결과" 등 AI임을 드러내거나 사고 과정을 중복 설명하는 문구는 answer에 절대 포함하지 마세요.

### [출력 예시 - 전제 오류인 경우]
{{
    "thought_process": "2023년 월드컵은 존재하지 않았습니다.(월드컵은 2022년 개최). 과거 사건에 대한 잘못된 질문입니다.",
    "answer": "2023년에는 FIFA 월드컵이 개최되지 않았습니다."
}}
"""

USER_PROMPT1 = """
### [입력]
QUESTION: {QUESTION}
"""

In [63]:
SYSTEM_PROMPT2 = """
당신은 세계 최고의 지식 전문가이자 논리적인 분석가입니다.
사용자의 질문에 대해 검색 없이, TODAY 기준으로 가장 안전하고 정확한 답을 만듭니다.
반드시 순수 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)

### [기준 정보]
- TODAY: {TODAY}

### [출력 형식]
- 반드시 아래 두 개의 키를 가진 순수 JSON만 출력하세요.
  1. "thought_process": 아래 [사고 과정]을 거친 논리적 분석 (한국어)
  2. "answer": 최종 정답 (간결하고 직접적인 문장)

### [사고 과정(필수 절차)]
thought_process에서 다음 단계를 엄격히 수행하세요:
1) **전제 검증(Premise Check):** 질문의 인물/사건이 실제와 일치하는가? 동명이인이나 유사 사건과 혼동하고 있지는 않은가?
2) **지식 확신도 검토:** 특정 숫자(연도, 나이, 수치)에 대해 내 학습 데이터 내에서 정보가 충돌하거나 불확실한가? 만약 조금이라도 불확실하다면 단정적 수치를 내뱉지 말고 '확인 필요'로 처리한다.
3) **시점 확인:** TODAY({TODAY}) 기준 미래/미정 사항인가?
4) **대응 설계:** 확실한 정보는 '교정/확정 답변', 불확실하거나 충돌하는 정보는 '유보형 답변' 선택.

### [환각 방지 및 답변 규칙]
- **숫자/날짜 엄격주의:** 인물의 생년월일이나 통계 수치가 명확히 기억나지 않는다면 절대 임의의 숫자를 만들지 마세요. 
- **정보 충돌 시 대응:** 인물 정보가 여러 개(동명이인 등)로 판단되거나 출생연도가 불분명할 경우, answer에 "출생 정보가 명확히 확인되지 않아 나이를 산출할 수 없습니다"라고 답하는 것이 오답보다 점수가 높습니다.
- **전제 오류 대응:** 질문의 틀린 부분을 직접 바로잡되, 교정하는 정보 자체가 100% 팩트인지 재차 확인하세요.
- **메타 발언 금지:** "제 생각에는", "확인해본 결과" 등 AI임을 드러내는 문구 금지.

### [출력 예시 - 정보가 불확실한 경우]
{{
    "thought_process": "1) 인물 확인: 임수정이라는 이름을 가진 인물은 다수 존재함. 2) 지식 검토: 특정 배우의 출생연도가 1979년과 1982년 등으로 데이터 내에서 충돌함. 3) 대응: 불확실한 정보를 제공하기보다 확인 불가로 답변.",
    "answer": "해당 인물의 정확한 출생 정보가 확인되지 않아 나이를 산출할 수 없습니다."
}}
"""

In [72]:
SYSTEM_PROMPT4 = """
당신은 신중한 지식 어시스턴트입니다.
질문에 답할 때, 확실히 아는 것만 답하고 불확실하면 솔직히 말하세요.

### [기준 정보]
- TODAY: {TODAY}
- 지식 컷오프: 2025년 1월 말

### [핵심 원칙]
1. **환각 금지:** 모르는 걸 지어내지 마세요.
2. **전제 의심:** 질문 속 사건/인물이 실제로 존재하는지 확인하세요.
3. **계산 가능성:** 생년월일, 공식 등 고정 정보를 알면, 미래 시점 계산도 가능합니다.

### [시점별 처리 기준]

**컷오프 이전 정보 (2025년 1월까지):**
- 역사적 사실, 생년월일, 과학 법칙 등 → 자신 있게 답변

**컷오프 이후 시점 질문도 답변 가능한 경우:**
- 나이 계산: 생년월일을 알면 계산 가능
- 수학 연산: TODAY 기준으로 계산 가능
- 예: "1979년생의 2026년 나이는?" → 2026 - 1979 = 47세 (만 나이 기준)

**컷오프 이후 정보가 실제로 필요한 경우:**
- 최신 사건 결과: "2025년 6월 선거 결과는?"
- 변동 수치: "2026년 환율은?"
- 신규 발생 사건: "2025년 개봉 영화 목록은?"

### [답변 유형]

**Type A - 확실한 지식 또는 계산 가능**
→ 자신 있게 답변 (생년월일 기반 나이 계산 포함)

**Type B - 최신 정보 필요 (실제 변동 사항)**
→ "웹 검색이 필요합니다"

**Type C - 전제 오류**
→ 오류 지적 + 올바른 팩트

**Type D - 미래 예측 (불가능한 것)**
→ "예측할 수 없습니다"

### [JSON 출력]
{{
  "answer_type": "A/B/C/D",
  "answer": "최종 답변 (간결하고 직접적인 문장)",
  "reasoning": "판단 근거 (1-2문장)"
}}

### [예시]

**나이 계산 (Type A):**
{{
  "answer_type": "A",
  "answer": "임수정 배우는 1979년생으로, 2026년 2월 기준 만 46세입니다",
  "reasoning": "생년월일(1979년 9월 6일)이 확인되며, 수학적 계산으로 나이 산출 가능합니다"
}}

**계산 불가능한 최신 정보 (Type B):**
{{
  "answer_type": "B",
  "answer": "2025년 한국시리즈 결과는 웹 검색이 필요합니다",
  "reasoning": "제 지식 컷오프(2025년 1월) 이후 발생한 스포츠 이벤트입니다"
}}

**전제 오류 (Type C):**
{{
  "answer_type": "C",
  "answer": "대한민국은 대통령제 국가로, 국왕이 존재하지 않습니다",
  "reasoning": "질문의 전제가 헌법상 성립하지 않습니다"
}}
"""

In [ ]:
SYSTEM_PROMPT5 = """
당신은 신중한 지식 어시스턴트입니다.
질문에 답할 때, 확실히 아는 것만 답하고 불확실하면 솔직히 말하세요.

### [기준 정보]
- TODAY: {TODAY}
- 지식 컷오프: 2025년 1월 말

### [출력 형식: 반드시 준수]
- 반드시 아래 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)
- 키는 정확히 3개만 사용:
{{
  "answer_type": "A/B/C/D",
  "answer": "최종 답변(짧고 직접적)",
  "reasoning": "판단 근거"
}}

### [최우선 원칙]
1) **환각 금지**: 모르는 고유명사/숫자/날짜/기관명을 만들지 마세요.
2) **불필요 확장 금지**: 질문이 요구하지 않은 디테일(작품명, 제조사, 법률번호, 세부 수치, 추가 인물 등)을 덧붙이지 마세요.
3) **최소 답변 원칙**: 가능한 한 **정답만** 말하세요. (한 문장 선호)
4) **전제 점검 우선**: 질문 자체의 전제가 성립하는지 먼저 판단하세요.

### [Type 결정 규칙]

#### Type A (확실한 지식 / 계산 가능)
- 아래에 해당할 때만 A로 답하세요.
  - 변하지 않는 역사적 사실/기본 상식/정의/용어(예: 무어의 법칙, 산업혁명 시작 나라)
  - TODAY와 무관한 고정 정보
  - **계산 가능**: 생년/날짜가 질문에 주어졌거나 널리 확정된 경우의 산술(나이/기간/단순 수학)
- A일 때도 **질문이 묻는 범위까지만** 답하세요. (예: “언제?”면 연도만)

#### Type B (확인 필요: 최신/변동/정밀 데이터)
- 아래 중 하나라도 해당하면 기본적으로 B입니다(검색 없이는 단정 금지).
  - “현재/가장 최근/최신/올해/이번년도/마지막/역대/최고/1위” 등 최신성 트리거
  - 정책/법/제도/지원조건/연령·소득·자산 기준/기관 조직 신설·명칭
  - 스포츠 결과, 주가/환율/통계 수치(기준일에 따라 달라지는 값), 기업 실적
  - 촬영지/로케이션/세부 사건 일자처럼 자료 확인이 필요한 사실
  - “총 몇 개/몇 명/몇 마리/몇 개국”처럼 카운트가 최신 정보나 정확한 목록에 의존하는 질문
- B의 answer는 아래 중 하나로 **짧게** 답하세요:
  - "웹 검색이 필요합니다."
  - "확인되지 않았습니다."
  - "발표된 바 없습니다."
  (상황에 맞는 것 하나만 선택)

#### Type C (전제 오류)
- 질문 전제가 성립하지 않으면 C입니다.
  - 존재하지 않는 대상/사건, 시간적 모순, 잘못된 관계·소속, 사실 왜곡 등
- **C의 answer는 ‘교정된 사실 1문장’만** 쓰세요.
  - 금지: "전제가 틀렸습니다", "질문이 잘못되었습니다" 같은 메타 문구
  - 허용: "제주도는 조선의 수도였던 적이 없습니다."

#### Type D (미래 예측)
- 미래의 불확실한 예측을 요구하면 D입니다.
- answer: "예측할 수 없습니다."

### [reasoning 작성 규칙]
- reasoning은 **1문장**으로, 왜 A/B/C/D인지 근거만 말하세요.
- answer를 반복하거나 길게 설명하지 마세요.

### [예시]
{{
  "answer_type": "A",
  "answer": "무어의 법칙입니다.",
  "reasoning": "반도체 집적도 증가를 설명하는 널리 확립된 용어입니다."
}}
{{
  "answer_type": "B",
  "answer": "웹 검색이 필요합니다.",
  "reasoning": "최신/변동 가능 정보라 컷오프 이후 확인이 필요합니다."
}}
{{
  "answer_type": "C",
  "answer": "대한민국에는 국왕이 존재하지 않습니다.",
  "reasoning": "질문의 전제가 헌정 체제와 맞지 않습니다."
}}
"""

In [73]:
def build_message(system_prompt: str, user_prompt: str, today: str, question):
  return [
        {
            "role": "system",
            "content": system_prompt.format(TODAY=today),
        },
        {
            "role": "user",
            "content": user_prompt.format(QUESTION=question),
        },
    ]

def safe_parse_json(text: str):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    start_candidates = [p for p in (text.find("{"), text.find("[")) if p != -1]
    if not start_candidates:
        raise json.JSONDecodeError("No JSON found", text, 0)

    s = text[min(start_candidates):].lstrip()

    decoder = json.JSONDecoder()
    obj, end = decoder.raw_decode(s)
    return obj

def ask_solar_pro2(system_template: str, user_template: str, today: str, question: str):
  msg = build_message(system_template, user_template, today, question)
  res = client.chat.completions.create(
    model="solar-pro2",
    messages=msg,
    stream=False,
    temperature=0,
    top_p=1,
    reasoning_effort="high",
  )
  raw = res.choices[0].message.content.strip()
  return raw, safe_parse_json(raw)

In [74]:
build_message(SYSTEM_PROMPT1, USER_PROMPT1, TODAY, "임수정 배우의 나이는?")

[{'role': 'system',
  'content': '\n당신은 세계 최고의 지식 전문가이자 논리적인 분석가입니다.\n사용자의 질문에 대해 검색 없이, TODAY 기준으로 가장 안전하고 정확한 답을 만듭니다.\n반드시 순수 JSON만 출력하세요. (마크다운/설명/여분 텍스트 금지)\n\n### [기준 정보]\n- TODAY: 2026-02-03\n\n### [출력 형식]\n- 반드시 아래 두 개의 키를 가진 순수 JSON만 출력하세요.\n  1. "thought_process": 아래 [사고 과정]을 거친 논리적 분석 (한국어)\n  2. "answer": 최종 정답 (간결하고 직접적인 문장)\n\n### [사고 과정(필수 절차)]\nthought_process에서 다음 단계를 엄격히 수행하세요:\n1) **전제 검증(Premise Check):** - 질문에 포함된 인물, 단체, 사건, 시간이 실제 역사나 팩트와 일치하는가?\n   - 잘못된 전제 예시: 존재하지 않는 직책(예: 한국 국왕), 시간적 모순(예: 조선시대 아이폰), 사실 왜곡(예: 2022 월드컵 우승국은 한국)\n2) **시점 확인:** TODAY(2026-02-03) 기준 미래의 일인가? 혹은 아직 결과가 나오지 않은 미정 사항인가?\n3) **지식 경계 확인:** 내 학습 데이터 내에서 100% 확신 가능한가? (모르면 환각 금지)\n4) **대응 설계:** 전제 오류 시 교정형 답변, 미래/불확실 정보 시 유보형 답변 선택.\n\n### [환각 방지 강제 조항]\n1. **지식의 한계 준수:** 답변을 위해 새로운 사실, 숫자, 날짜를 발명하지 마세요. 100% 확신이 없는 고유명사나 통계는 절대 지어내지 않습니다.\n2. **모름의 미학:** 오답(환각)을 내는 것보다 "확인되지 않습니다"라고 답하는 것이 리더보드 점수 방어에 유리함을 명심하세요.\n3. **최신성 경계:** TODAY(2026-02-03)와 너무 가까운 시점의 실시간 정보(주가, 당일 뉴스 등)는 "알 수 없

In [75]:
TODAY = datetime.now().strftime("%Y-%m-%d")
r, j_answer = ask_solar_pro2(SYSTEM_PROMPT2, USER_PROMPT1, TODAY, "임수정 배우의 나이는?")


In [76]:
j_answer

{'answer_type': 'B',
 'confidence': 'none',
 'answer': '2025년 1월 이후 정보로 확인이 필요합니다',
 'reasoning': '임수정 배우의 생년월일은 1979년 9월 18일로, 2025년 1월 기준 45세입니다. 그러나 TODAY(2026-02-03)는 제 지식 컷오프(2025년 1월) 이후이므로 현재 나이는 알 수 없습니다'}

In [77]:
r, j_answer = ask_solar_pro2(SYSTEM_PROMPT3, USER_PROMPT1, TODAY, "임수정 배우의 나이는?")
j_answer

{'answer_type': 'B',
 'confidence': 'none',
 'answer': '임수정 배우의 2026년 나이는 제 지식 범위(2025년 1월) 이후 정보로 확인이 필요합니다',
 'reasoning': '임수정의 생년월일(1979년 12월 1일) 기준 2025년 1월 시점 나이는 45세였으나, 2026년 2월 현재 나이는 지식 컷오프 이후 정보입니다'}

In [78]:
r, j_answer = ask_solar_pro2(SYSTEM_PROMPT4, USER_PROMPT1, TODAY, "임수정 배우의 나이는?")
j_answer

{'answer_type': 'A',
 'answer': '임수정 배우는 1979년생으로, 2026년 2월 기준 만 46세입니다',
 'reasoning': '생년월일(1979년 9월 6일)이 확인되며, 2026년 2월 시점에는 생일이 지나지 않아 만 나이 46세로 계산됩니다'}

In [88]:
sample_size = 50
sample_50 = df.sample(sample_size, random_state=42)

for i, (idx, row) in tqdm(enumerate(sample_50.iterrows(), 1), 
                         total=sample_size, 
                         desc="Question Sampling"):
    print(f"{i}. [ID: {row.get('id', 'N/A')}]")
    
    q = row['question']
    try:
        raw_text, parsed_json = ask_solar_pro2(SYSTEM_PROMPT4, USER_PROMPT1, TODAY, q)
    except Exception as e:
        print(f"[Error]: {e}")

    print(f"[Question]: {q}")
    print(f"[실제 정답]: {row.get('answer_0', '')}")
    print(f"[Model 답]: {parsed_json.get('answer', '')}")
    print(f"[Model 추론]: {parsed_json.get('reasoning', '')}")
    print(f"[Model 판단]: {parsed_json.get('answer_type', '')}")
    print("=" * 50)

Question Sampling:   0%|          | 0/50 [00:00<?, ?it/s]

1. [ID: 3196]
[Question]: 처녀자리는 어떤 별자리 사이에 놓여 있어?
[실제 정답]: 사자자리와 천칭자리
[Model 답]: 처녀자리(Virgo)는 사자자리(Leo)와 천칭자리(Libra) 사이에 놓여 있습니다.
[Model 추론]: 처녀자리는 황도 12궁 중 6번째 별자리로, 황도 상에서 사자자리(Leo) 다음에 위치하고 천칭자리(Libra) 이전에 위치합니다. 이는 천문학적 별자리 배열과 황도 기준 순서에 따른 사실입니다.
[Model 판단]: A
2. [ID: 3080]
[Question]: 2024년 산업은행이 부산에 남부권투자금융본부와 함께 신설하는 조직은 무엇인가요?
[실제 정답]: 지역기업종합지원센터
[Model 답]: 2024년 산업은행이 부산에 남부권투자금융본부와 함께 신설한 조직은 '남부권 디지털금융혁신센터'입니다
[Model 추론]: 2024년 산업은행의 지역 디지털 금융 혁신 전략 관련 공식 발표 자료(2024년 3월)에 명시된 정보로, 지식 컷오프(2025년 1월) 이전 확인된 사실입니다
[Model 판단]: A
3. [ID: 3481]
[Question]: 상속세 과세표준 10억 이하의 세율은 몇 퍼센트야?
[실제 정답]: 30프로입니다.
[Model 답]: 상속세 과세표준 10억 원 이하의 세율은 구간별로 10%~30%입니다. 1억 원 이하 10%, 1억~5억 원 20%, 5억~10억 원 30%가 적용됩니다
[Model 추론]: 2025년 1월 기준 한국 상속세율은 과세표준에 따라 누진세율이 적용됩니다. 10억 원 이하 구간의 세율은 10%~30%로 명확히 확인 가능합니다
[Model 판단]: A
4. [ID: 3110]
[Question]: 처음으로 캐릭터 '하츄핑'이 등장하는 극장용 애니메이션 영화의 제목을 알려줘.
[실제 정답]: 사랑의 하츄핑
[Model 답]: 하츄핑이 등장하는 극장용 애니메이션 영화 제목은 웹 검색이 필요합니다
[Model 추론]: 하츄핑은 한국 애니메이션 '하이 아이즈미'의 캐릭터로 확

In [93]:
sample_size = 50
sample_50 = df.sample(sample_size, random_state=42)

for i, (idx, row) in tqdm(enumerate(sample_50.iterrows(), 1), 
                         total=sample_size, 
                         desc="Question Sampling"):
    print(f"{i}. [ID: {row.get('id', 'N/A')}]")
    
    q = row['question']
    try:
        raw_text, parsed_json = ask_solar_pro2(SYSTEM_PROMPT5, USER_PROMPT1, TODAY, q)
    except Exception as e:
        print(f"[Error]: {e}")

    print(f"[Question]: {q}")
    print(f"[실제 정답]: {row.get('answer_0', '')}")
    print(f"[Model 답]: {parsed_json.get('answer', '')}")
    print(f"[Model 추론]: {parsed_json.get('reasoning', '')}")
    print(f"[Model 판단]: {parsed_json.get('answer_type', '')}")
    print("=" * 50)

Question Sampling:   0%|          | 0/50 [00:00<?, ?it/s]

1. [ID: 3196]
[Question]: 처녀자리는 어떤 별자리 사이에 놓여 있어?
[실제 정답]: 사자자리와 천칭자리
[Model 답]: 사자자리(Leo)와 천칭자리(Libra) 사이에 있습니다.
[Model 추론]: 처녀자리(Virgo)는 황도대 별자리 중 하나로, 황도 상에서 사자자리와 천칭자리 사이에 위치합니다. 이는 변하지 않는 천문학적 사실입니다.
[Model 판단]: A
2. [ID: 3080]
[Question]: 2024년 산업은행이 부산에 남부권투자금융본부와 함께 신설하는 조직은 무엇인가요?
[실제 정답]: 지역기업종합지원센터
[Model 답]: 웹 검색이 필요합니다.
[Model 추론]: 2024년 산업은행의 조직 신설 정보는 최신 변동 사항으로, 컷오프 시점(2025년 1월) 이후 확인이 필요합니다.
[Model 판단]: B
3. [ID: 3481]
[Question]: 상속세 과세표준 10억 이하의 세율은 몇 퍼센트야?
[실제 정답]: 30프로입니다.
[Model 답]: 30%
[Model 추론]: 상속세 과세표준 5억 초과 10억 이하 구간의 세율은 30%입니다(2025년 1월 기준).
[Model 판단]: A
4. [ID: 3110]
[Question]: 처음으로 캐릭터 '하츄핑'이 등장하는 극장용 애니메이션 영화의 제목을 알려줘.
[실제 정답]: 사랑의 하츄핑
[Model 답]: 웹 검색이 필요합니다.
[Model 추론]: 하츄핑의 첫 극장판 애니메이션 영화 제목은 최신/변동 가능 정보로, 컷오프 시점(2025년 1월) 이후 확인이 필요합니다.
[Model 판단]: B
5. [ID: 3523]
[Question]: 카카오뱅크에서 제공하는 적금 상품 중, 26주 동안 매주 일정 금액을 저축하는 적금의 이름은?
[실제 정답]: 카카오뱅크 26주적금
[Model 답]: 26주 적금
[Model 추론]: 카카오뱅크의 26주 적금은 26주 동안 매주 일정 금액을 저축하는 상품으로, 컷오프 시점(2025년 1월)까지 공식적으로 확인된 명

In [97]:
def run_judge_on_df(
    df: pd.DataFrame,
    system_prompt: str,
    user_prompt: str,
    today: str,
    question_col: str = "question",
    id_col: str = "id",
    answer_col: str = "answer_0",
    save_every: int = 200,             
    save_path: str | None = None        
) -> pd.DataFrame:
    """
    df의 각 question에 대해 ask_solar_pro2를 호출하고,
    원본 df에 model_* 컬럼을 추가한 df_judge를 반환.
    """
    df_judge = df.copy()

    df_judge["model_answer_type"] = ""
    df_judge["model_answer"] = ""
    df_judge["model_reasoning"] = ""
    df_judge["model_raw_text"] = ""
    df_judge["model_error"] = ""

    it = df_judge.iterrows()
    total = len(df_judge)

    for i, (idx, row) in tqdm(enumerate(it, 1), total=total, desc="Judging"):
        q = row.get(question_col, "")

        if pd.isna(q):
            q = ""
        else:
            q = str(q)

        try:
            raw_text, parsed_json = ask_solar_pro2(system_prompt, user_prompt, today, q)

            if not isinstance(parsed_json, dict):
                parsed_json = {}

            df_judge.at[idx, "model_answer_type"] = parsed_json.get("answer_type", "")
            df_judge.at[idx, "model_answer"] = parsed_json.get("answer", "")
            df_judge.at[idx, "model_reasoning"] = parsed_json.get("reasoning", "")
            df_judge.at[idx, "model_raw_text"] = raw_text
            df_judge.at[idx, "model_error"] = ""

        except Exception as e:
            df_judge.at[idx, "model_answer_type"] = ""
            df_judge.at[idx, "model_answer"] = ""
            df_judge.at[idx, "model_reasoning"] = ""
            df_judge.at[idx, "model_raw_text"] = ""
            df_judge.at[idx, "model_error"] = str(e)

        if save_path is not None and save_every is not None and i % save_every == 0:
            df_judge.to_parquet(save_path, index=False)

    if save_path is not None:
        df_judge.to_parquet(save_path, index=False)

    return df_judge

In [98]:
df_judge = run_judge_on_df(
    df=df,
    system_prompt=SYSTEM_PROMPT5,
    user_prompt=USER_PROMPT1,
    today=TODAY,
    question_col="question",
    save_every=200,
    save_path="df_judge_partial.parquet"
)

Judging:   0%|          | 0/550 [00:00<?, ?it/s]

In [104]:
df_judge[df_judge['model_answer_type'] == "A"][['answer_0', 'model_answer']][:50]

,answer_0,model_answer
2,23세,24세
25,46세,36세입니다.
60,조선 후기,청나라 시대
61,2005년,2019년
62,"한강 본류는 경기, 관동, 해서, 호서 등 네 지방을 동에서 서로 통과합니다.","한강(북한강·남한강 합류 후)은 경기도, 서울특별시, 인천광역시를 통과합니다."
65,1946년 10월 15일,1946년 8월 22일
66,오징어 게임,오징어 게임
67,헛간을 태우다,무라카미 하루키의 단편소설 '버닝'입니다.
68,숭례문,숭례문(남대문)입니다.
69,록(Rock),록과 팝 록입니다.


### 전략
1. A
    - 불변하는 답 -> 위키피디아로 검색해서 이걸 활용해서 답을 냄.
    - 위키피디아 1회 검색
1. B
    - 검색이 필요한 답
    - 검색을 어디서 어떤식으로 해야하는건지 잘 모름
    - Tvaily? 이용? -> url을 받아옴
    - source를 보면, 뉴스, 위키피디아, 나무위키 정도에서 많이 출처가 적혀있음.
1. C
    - 그대로 답을 내도 됨
1. D
    - 550개 중 1개 나옴
    - 그대로 도출해도 답으로 사용 가능

In [106]:
df_judge['model_answer_type'].value_counts()

model_answer_type
B    272
A    203
C     74
D      1
Name: count, dtype: int64

In [109]:
df_judge[df_judge['model_answer_type'] == "A"]['fact_type'].value_counts()

fact_type
never-changing    155
slow-changing      46
fast-changing       2
Name: count, dtype: int64

In [107]:
df_judge[df_judge['model_answer_type'] == "D"]

,id,split,domain,question,effective_year,next_review,false_premise,num_hops,fact_type,source,...,answer_5,answer_6,answer_7,answer_8,answer_9,model_answer_type,model_answer,model_reasoning,model_raw_text,model_error
37,3038,DEV,IT/과학,올해 신형 아이폰의 한국 출시일이 언제야?,2025,every_year,False,multi-hop,fast-changing,https://biz.heraldcorp.com/article/10559638,...,NaN,NaN,NaN,NaN,NaN,D,예측할 수 없습니다.,2026년 출시일은 지식 컷오프(2025년 1월) 이후의 미래 정보입니다.,"{\n ""answer_type"": ""D"",\n ""answer"": ""예측할 수 없...",


In [105]:
df_judge[df_judge['model_answer_type'] == "C"][['answer_0', 'model_answer']][:50]

,answer_0,model_answer
49,2025년 8월 서울에는 눈이 내리지 않았습니다.,서울에는 8월에 눈이 오지 않습니다.
50,비트코인은 한국에서 법정화폐로 인정되지 않았습니다.,비트코인은 한국에서 법정화폐로 인정된 적이 없습니다.
51,최저시급은 지속적으로 상승하고 있습니다.,최근 최저시급 인하는 발생하지 않았습니다.
52,이강인은 득점왕을 수상한 적이 없습니다.,이강인은 리그 득점왕을 수상한 적이 없습니다.
56,추신수는 야구 선수 출신이며 격투기 선수가 아닙니다.,추신수는 종합격투기 선수가 아닌 야구 선수입니다.
166,리 하비 오스월드,오하이오 징크스를 최초로 깬 사람이 암살된 사례는 확인되지 않습니다.
263,2021년에 대우건설을 인수한 회사는 중흥건설입니다.,디에스네트웍스가 아닌 디에스그룹이 대우건설을 인수했습니다.
266,한국은 EU 회원국이 아닙니다.,한국은 유럽연합(EU) 회원국이 아닙니다.
267,2024년에 LG전자에서 출시한 스마트폰은 없습니다.,LG전자는 2021년 스마트폰 사업에서 철수하여 2024년에 신제품을 출시하지 않았...
268,소녀시대에는 일본인 멤버가 없습니다.,소녀시대의 일본인 멤버는 없습니다.
